# ESN + Ridge Regression Baseline

Ziel: Mackey-Glass-Zeitreihe (tau=17) mit einem Echo State Network
und Ridge Regression als Readout vorhersagen. Dieses Notebook
etabliert die Baseline für den späteren Vergleich mit HDC.

Parameter:
- Reservoir: 500 Neuronen, spectral_radius=0.9, sparsity=0.1
- Warmup: 100 Schritte verwerfen
- Regularisierung: alpha=1e-6

In [1]:
import matplotlib
matplotlib.use('Agg')
from src.data.mackey_glass import generate_mackey_glass, prepare_dataset
from src.reservoir.esn import EchoStateNetwork
from src.baselines.ridge_readout import RidgeReadout
import numpy as np
import matplotlib.pyplot as plt
import time

In [2]:
# Mackey-Glass Zeitreihe mit Standardparametern
daten = generate_mackey_glass(n_steps=12000, tau=17)

# 80/20 Split — X sind Inputs, y ist der nächste Zeitschritt
X_train, y_train, X_test, y_test = prepare_dataset(daten, train_ratio=0.8)

print(f"Trainings-Samples: {len(X_train)}")
print(f"Test-Samples:      {len(X_test)}")

# Erste 500 Schritte visualisieren
plt.figure(figsize=(12, 3))
plt.plot(daten[:500])
plt.title("Mackey-Glass Zeitreihe (erste 500 Schritte)")
plt.xlabel("Zeitschritt")
plt.ylabel("x(t)")
plt.tight_layout()
plt.show()

Trainings-Samples: 9599
Test-Samples:      2400


C:\Users\lfLaw\AppData\Local\Temp\ipykernel_55504\1532398531.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ESN initialisieren
esn = EchoStateNetwork(n_reservoir=500, spectral_radius=0.9, sparsity=0.1, seed=42)

# Trainingsdaten durch das Reservoir schicken
t0 = time.time()
train_states = esn.run(X_train)
t_reservoir_train = time.time() - t0

# Warmup verwerfen: erste 100 Zustände sind Einschwingzeit
warmup = 100
train_states_cut = train_states[warmup:]
y_train_cut = y_train[warmup:]

# Reservoir für Testdaten zurücksetzen und neu durchlaufen
esn.reset()
test_states = esn.run(X_test)

print(f"Reservoir-Durchlauf (Training): {t_reservoir_train:.2f}s")
print(f"Trainings-Zustände nach Warmup: {train_states_cut.shape}")
print(f"Test-Zustände:                  {test_states.shape}")

Reservoir-Durchlauf (Training): 0.35s
Trainings-Zustände nach Warmup: (9499, 500)
Test-Zustände:                  (2400, 500)


In [4]:
# Ridge Readout trainieren
readout = RidgeReadout(alpha=1e-6)

t0 = time.time()
readout.fit(train_states_cut, y_train_cut)
t_training = time.time() - t0

# Vorhersage auf Testdaten
y_pred = readout.predict(test_states)
nrmse = readout.score(test_states, y_test)

print(f"Trainingszeit (Ridge): {t_training*1000:.1f}ms")
print(f"NRMSE auf Testdaten:   {nrmse:.4f}")

if nrmse < 0.1:
    print("Ergebnis im erwarteten Bereich (< 0.1)")
else:
    print("NRMSE > 0.1 -- pruefe Implementierung")

Trainingszeit (Ridge): 30.7ms
NRMSE auf Testdaten:   1.1475
NRMSE > 0.1 -- pruefe Implementierung


In [5]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Plot 1: Vorhersage vs. tatsächliche Werte
n_plot = 300
axes[0].plot(y_test[:n_plot], label="Tatsaechlich", linewidth=1.5)
axes[0].plot(y_pred[:n_plot], label="Vorhersage", linewidth=1.5, linestyle="--")
axes[0].set_title(f"ESN + Ridge Regression -- erste {n_plot} Test-Schritte")
axes[0].set_xlabel("Zeitschritt")
axes[0].set_ylabel("x(t)")
axes[0].legend()

# Plot 2: Vorhersagefehler über die Zeit
fehler = np.abs(y_pred[:n_plot] - y_test[:n_plot])
axes[1].plot(fehler, color="red", linewidth=1, label="Absoluter Fehler")
axes[1].set_title("Vorhersagefehler ueber die Zeit")
axes[1].set_xlabel("Zeitschritt")
axes[1].set_ylabel("|y_pred - y_test|")
axes[1].legend()

plt.tight_layout()
plt.show()

C:\Users\lfLaw\AppData\Local\Temp\ipykernel_55504\349039007.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ergebnisse

| Metrik | Wert |
|--------|------|
| NRMSE | *wird nach Ausführung eingetragen* |
| Trainingszeit (Ridge) | *ms* |
| Reservoir-Durchlauf | *s* |

## Interpretation

Ein NRMSE < 0.1 bedeutet, dass das ESN die chaotische Mackey-Glass-Dynamik
erfolgreich gelernt hat. Ridge Regression als linearer Readout ist ausreichend,
weil das Reservoir die nichtlineare Transformation übernimmt.

## Offene Fragen
- Wie reagiert das System auf andere Spektralradien (0.5, 1.2)?
- Nächster Schritt: HDC-Readout als Alternative zu Ridge Regression